In [1]:
import os
# Must be set BEFORE CUDA / torch initializes to avoid memory fragmentation OOM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
#!rm -rf /kaggle/working/SwinLLIE


In [2]:
!pip install -q fastapi uvicorn python-multipart pyngrok pillow timm einops
!pip install -q scikit-image   # for real PSNR/SSIM


In [3]:
import os, sys, subprocess

# ---- Clone SwinLLIE from your GitHub ----------
REPO_URL = "https://github.com/kavindamihiran/SwinLLIE.git"
BRANCH   = "final"

# Ensure we're in a valid directory
os.makedirs("/kaggle/working", exist_ok=True)
os.chdir("/kaggle/working")

if not os.path.exists("/kaggle/working/SwinLLIE"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL,
                    "SwinLLIE"], check=True)

sys.path.insert(0, "/kaggle/working/SwinLLIE")
os.chdir("/kaggle/working/SwinLLIE")

In [4]:
import sys
sys.path.insert(0, "/home/kavinda/Desktop/project/SwinLLIE")

import torch
from swinllie import SwinLLIE

CHECKPOINT_PATH = "./experiments/test_run/checkpoints/best.pth"
WINDOW_SIZE     = 8
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

model = SwinLLIE()
ckpt  = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
model = model.to(DEVICE)
model.eval()
print("✅ Model loaded!")

Using device: cuda
✅ Model loaded!


In [5]:
import io, uuid, time, base64
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity, peak_signal_noise_ratio

import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import Optional

# ----------------------------------------
# Metric Helpers (from swinllie/utils.py 
# -----------------------------------------
def _detect_data_range(img1, img2, data_range=None):
    if data_range is not None:
        return float(data_range)
    if img1.dtype == np.uint8 or img2.dtype == np.uint8:
        return 255.0
    max_val = max(img1.max(), img2.max())
    min_val = min(img1.min(), img2.min())
    if max_val > 1.0 or min_val < 0.0:
        if max_val > 1.0 and max_val <= 255.0:
            return 255.0
        return max_val - min_val if max_val > min_val else 1.0
    return 1.0


def calculate_psnr(img1, img2, crop_border=0, data_range=None):
    assert img1.shape == img2.shape
    detected_range = _detect_data_range(img1, img2, data_range)
    if crop_border > 0:
        h, w = img1.shape[:2]
        img1 = img1[crop_border:-crop_border, crop_border:-crop_border, ...]
        img2 = img2[crop_border:-crop_border, crop_border:-crop_border, ...]
    return peak_signal_noise_ratio(img1.astype(np.float64), img2.astype(np.float64),
                                   data_range=detected_range)


def calculate_ssim(img1, img2, crop_border=0, data_range=None):
    assert img1.shape == img2.shape
    detected_range = _detect_data_range(img1, img2, data_range)
    if crop_border > 0:
        h, w = img1.shape[:2]
        img1 = img1[crop_border:-crop_border, crop_border:-crop_border, ...]
        img2 = img2[crop_border:-crop_border, crop_border:-crop_border, ...]
    img1, img2 = img1.astype(np.float64), img2.astype(np.float64)
    h, w = img1.shape[:2]
    win_size = min(11, h if h % 2 == 1 else h - 1, w if w % 2 == 1 else w - 1)
    win_size = max(3, win_size)
    if len(img1.shape) == 3 and img1.shape[2] == 3:
        return structural_similarity(img1, img2, data_range=detected_range,
                                     channel_axis=2, win_size=win_size,
                                     gaussian_weights=True, sigma=1.5, K1=0.01, K2=0.03)
    img1 = img1.squeeze(); img2 = img2.squeeze()
    return structural_similarity(img1, img2, data_range=detected_range,
                                 win_size=win_size, gaussian_weights=True,
                                 sigma=1.5, K1=0.01, K2=0.03)


# -------------------------------------------------------
# Tile-based inference  — no downscaling, no OOM
# -------------------------------------------------------
# Tune TILE_SIZE downward (e.g. 256) if you still hit OOM.
# The model pads each tile internally so any size works.
TILE_SIZE    = 512   # pixels per tile edge
TILE_OVERLAP = 32    # overlap to hide seam artefacts


def tile_inference(x: "torch.Tensor", mdl, tile=TILE_SIZE, overlap=TILE_OVERLAP):
    """
    Process a large image tile-by-tile to avoid CUDA OOM.
    No resolution change — every tile goes through the model at full quality.
    Overlapping tiles are averaged so seams are invisible.
    """
    import torch
    B, C, H, W = x.shape
    stride = tile - overlap

    def get_starts(dim):
        if dim <= tile:
            return [0]
        starts = list(range(0, dim - tile, stride))
        if starts[-1] + tile < dim:
            starts.append(dim - tile)
        return starts

    h_starts = get_starts(H)
    w_starts = get_starts(W)

    output = torch.zeros(B, C, H, W, dtype=x.dtype, device=x.device)
    weight = torch.zeros(B, 1, H, W, dtype=x.dtype, device=x.device)

    with torch.no_grad():
        for hs in h_starts:
            he = min(hs + tile, H)
            for ws in w_starts:
                we = min(ws + tile, W)

                tile_in  = x[:, :, hs:he, ws:we]
                # model.forward() already calls check_image_size() internally,
                # so it pads + crops automatically — output equals input shape.
                tile_out = mdl(tile_in)

                output[:, :, hs:he, ws:we] += tile_out
                weight[:, :, hs:he, ws:we] += 1.0

                # Free intermediate CUDA allocations after each tile
                if x.device.type == "cuda":
                    torch.cuda.empty_cache()

    return output / weight.clamp(min=1e-8)


def tensor_to_pil(t):
    arr = t.squeeze(0).permute(1, 2, 0).cpu().numpy()
    arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def pil_to_tensor(img):
    arr = np.array(img).astype(np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)


def enhance_image_bytes(img_bytes: bytes) -> "tuple[bytes, float, float, float]":
    """
    Run SwinLLIE on raw image bytes using tiled inference.
    Returns (enhanced_bytes, psnr, ssim, elapsed_seconds).
    """
    start = time.time()

    orig = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    x    = pil_to_tensor(orig).to(DEVICE)

    # Tile-based inference — full model quality, bounded VRAM usage
    out = tile_inference(x, model, tile=TILE_SIZE, overlap=TILE_OVERLAP)

    enhanced = tensor_to_pil(out)

    orig_arr     = np.array(orig)
    enhanced_arr = np.array(enhanced)   # already same size as orig

    p = round(float(calculate_psnr(orig_arr, enhanced_arr)), 2)
    s = round(float(calculate_ssim(orig_arr, enhanced_arr)), 4)

    buf = io.BytesIO()
    enhanced.save(buf, format="PNG")
    elapsed = round(time.time() - start, 2)

    return buf.getvalue(), p, s, elapsed


# -------------------------------------------------------
# FastAPI
# -------------------------------------------------------
app = FastAPI(title="SwinLLIE API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

SESSIONS: dict[str, bytes] = {}

class QualityRequest(BaseModel):
    session_id: str

@app.get("/")
def root():
    return {"status": "ok", "gpu": DEVICE}

@app.get("/health")
def health():
    return {"status": "ok", "gpu": DEVICE,
            "cuda_name": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "N/A"}

@app.post("/api/enhance/fast")
async def enhance_fast(image: UploadFile = File(...)):
    allowed = {"image/jpeg", "image/jpg", "image/png"}
    if image.content_type not in allowed:
        raise HTTPException(400, "Only JPG / PNG allowed.")

    contents = await image.read()
    if len(contents) > 10 * 1024 * 1024:
        raise HTTPException(400, "File too large (max 10 MB).")

    session_id = str(uuid.uuid4())
    SESSIONS[session_id] = contents

    try:
        enhanced_bytes, p, s, elapsed = enhance_image_bytes(contents)
    except Exception as e:
        raise HTTPException(500, str(e))

    orig_b64     = base64.b64encode(contents).decode()
    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()

    return {
        "success":         True,
        "session_id":      session_id,
        "original_url":    f"data:image/png;base64,{orig_b64}",
        "image_url":       f"data:image/png;base64,{enhanced_b64}",
        "psnr":            p,
        "ssim":            s,
        "mode":            "fast",
        "processing_time": elapsed,
    }

@app.post("/api/enhance/quality")
async def enhance_quality(req: QualityRequest):
    contents = SESSIONS.get(req.session_id)
    if contents is None:
        raise HTTPException(404, "Session not found. Please upload image again.")

    try:
        enhanced_bytes, p, s, elapsed = enhance_image_bytes(contents)
    except Exception as e:
        raise HTTPException(500, str(e))

    enhanced_b64 = base64.b64encode(enhanced_bytes).decode()

    return {
        "success":         True,
        "session_id":      req.session_id,
        "image_url":       f"data:image/png;base64,{enhanced_b64}",
        "psnr":            p,
        "ssim":            s,
        "mode":            "quality",
        "processing_time": elapsed,
    }


In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

NGROK_AUTH_TOKEN = ""
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

ngrok.kill()
public_url = ngrok.connect(8000)

print("=" * 60)
print(f"  🚀 Backend URL: {public_url}")
print("  Copy this URL into .env.local as NEXT_PUBLIC_BACKEND_URL")
print("=" * 60)
print("  Uvicorn starting — live request logs will appear below:")
print("  (Stop this cell to shut down the server)")
print("=" * 60)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


  🚀 Backend URL: NgrokTunnel: "https://uncheerful-apheliotropically-lizbeth.ngrok-free.dev" -> "http://localhost:8000"
  Copy this URL into .env.local as NEXT_PUBLIC_BACKEND_URL
  Uvicorn starting — live request logs will appear below:
  (Stop this cell to shut down the server)
INFO:     158.178.231.254:0 - "OPTIONS /api/enhance/fast HTTP/1.1" 200 OK
INFO:     158.178.231.254:0 - "POST /api/enhance/fast HTTP/1.1" 200 OK
INFO:     158.178.231.254:0 - "OPTIONS /api/enhance/quality HTTP/1.1" 200 OK
INFO:     158.178.231.254:0 - "POST /api/enhance/quality HTTP/1.1" 200 OK
